<a href="https://colab.research.google.com/github/pranuk050-pixel/Pyspark_Programming/blob/main/2026_Q3_DF1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StructType, StringType, DateType, IntegerType
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Retail_dat_processing").getOrCreate()

In [ ]:
# 1. Read the source CSV. --> done

# Display:

# First 10 records --> done
# Schema  --> done
# Number of records --> done
sales_schema = StructType([ StructField('order_id', StringType()),
                          StructField('order_date', DateType()),
                          StructField('customer_id', StringType()),
                          StructField('customer_name', StringType()),
                          StructField('product_id', StringType()),
                          StructField('product_name', StringType()),
                          StructField('category', StringType()),
                          StructField('store_id', StringType()),
                          StructField('store_name', StringType()),
                          StructField('city', StringType()),
                          StructField('sate', StringType()),
                          StructField('quantity', IntegerType()),
                          StructField('unit_price', IntegerType()),
                          StructField('discount_pct', IntegerType())])


input_df = spark.read.csv('sales_data.csv', header= True, schema=sales_schema)

# input_df = spark.read.csv('sales_data.csv', header= True, inferSchema=True)
# in prod, we should not use inferSchema=True for csv file ==> does not give good performance

input_df.show(10, False)
input_df.printSchema()
input_df.count()

+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+
|order_id  |order_date|customer_id|customer_name  |product_id|product_name|category   |store_id|store_name   |city     |sate       |quantity|unit_price|discount_pct|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+
|ORD0000001|2026-05-31|CUST22129  |James Brown    |P003      |Headphones  |Electronics|S005    |Main Store   |Pune     |Maharashtra|8       |3000      |5           |
|ORD0000002|2026-03-07|CUST29205  |Mia Thomas     |P005      |Office Chair|Furniture  |S005    |Main Store   |Pune     |Maharashtra|9       |7500      |5           |
|ORD0000003|2026-03-29|CUST19584  |Daniel Martin  |P010      |Bookshelf   |Furniture  |S003    |City Store   |Chennai  |Tamil Nadu |10      |6000      |5           |
|ORD

444258

In [ ]:
# 2. Create a new column called gross_amount. --> done
# Formula: quantity × unit_price

gross_df = input_df.withColumn("gross_amount", input_df.quantity * input_df.unit_price)
gross_df.show(10, False)
gross_df.printSchema()

+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+
|order_id  |order_date|customer_id|customer_name  |product_id|product_name|category   |store_id|store_name   |city     |sate       |quantity|unit_price|discount_pct|gross_amount|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+
|ORD0000001|2026-05-31|CUST22129  |James Brown    |P003      |Headphones  |Electronics|S005    |Main Store   |Pune     |Maharashtra|8       |3000      |5           |24000       |
|ORD0000002|2026-03-07|CUST29205  |Mia Thomas     |P005      |Office Chair|Furniture  |S005    |Main Store   |Pune     |Maharashtra|9       |7500      |5           |67500       |
|ORD0000003|2026-03-29|CUST19584  |Daniel Martin  |P010      |Bookshelf   |Furniture  |S003    |City Stor

In [ ]:
# 3. Create discount_amount.
# Formula: gross_amount * discount_pct / 100

discount_df = gross_df.withColumn('discount_amount',  gross_df.gross_amount * gross_df.discount_pct / 100)
discount_df.show(10, False)

+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+
|order_id  |order_date|customer_id|customer_name  |product_id|product_name|category   |store_id|store_name   |city     |sate       |quantity|unit_price|discount_pct|gross_amount|discount_amount|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+
|ORD0000001|2026-05-31|CUST22129  |James Brown    |P003      |Headphones  |Electronics|S005    |Main Store   |Pune     |Maharashtra|8       |3000      |5           |24000       |1200.0         |
|ORD0000002|2026-03-07|CUST29205  |Mia Thomas     |P005      |Office Chair|Furniture  |S005    |Main Store   |Pune     |Maharashtra|9       |7500      |5           |67500       |3375.0         |
|ORD0000003|2026-03-29|CU

In [ ]:
# 4. Create a new column final_amount.
# Formula: gross_amount - discount_amount

final_df = discount_df.withColumn('final_amount', discount_df.gross_amount - discount_df.discount_amount)
final_df.show(10, False)

+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+------------+
|order_id  |order_date|customer_id|customer_name  |product_id|product_name|category   |store_id|store_name   |city     |sate       |quantity|unit_price|discount_pct|gross_amount|discount_amount|final_amount|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+------------+
|ORD0000001|2026-05-31|CUST22129  |James Brown    |P003      |Headphones  |Electronics|S005    |Main Store   |Pune     |Maharashtra|8       |3000      |5           |24000       |1200.0         |22800.0     |
|ORD0000002|2026-03-07|CUST29205  |Mia Thomas     |P005      |Office Chair|Furniture  |S005    |Main Store   |Pune     |Maharashtra|9       |7500      |5           |675

In [ ]:
# 5. Rename columns.

# customer_id     → customer_key
# product_id      → product_key
# store_id        → store_key
# unit_price      → selling_price
# discount_pct    → discount_percentage

# renamed_df = final_df.withColumnRenamed('customer_id', 'customer_key') \
#                       .withColumnRenamed('product_id', 'product_key')  \
#                       .withColumnRenamed('store_id', 'store_key') \
#                       .withColumnRenamed('unit_price', 'selling_price') \
#                       .withColumnRenamed('discount_pct', 'discount_percentage')

renamed_df = final_df.withColumnsRenamed({'customer_id' : 'customer_key',
                                          'product_id' : 'product_key',
                                          'store_id': 'store_key',
                                          'unit_price': 'selling_price',
                                          'discount_pct': 'discount_percentage'})

renamed_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_key: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- store_key: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- sate: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- selling_price: integer (nullable = true)
 |-- discount_percentage: integer (nullable = true)
 |-- gross_amount: integer (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- final_amount: double (nullable = true)



In [ ]:
# 6. Create store-level sales summary.

# For every store calculate:

# total_orders
# total_quantity
# total_gross_sales
# total_discount
# total_sales
# average_sales
# minimum_sales
# maximum_sales

store_level_summary_df = renamed_df.groupBy("store_key").agg(
    count('order_id').alias("total_orders"),
    sum('quantity').alias("total_quantity"),
    sum('gross_amount').alias("totalgross_sales"),
    sum('discount_amount').alias("total_discount"),
    sum('final_amount').alias("total_sales"),
    avg('final_amount').alias("average_sales"),
    min('final_amount').alias("minimum_sales"),
    max('final_amount').alias("maximum_sales"),
)

store_level_summary_df.show()

+---------+------------+--------------+----------------+--------------+-------------+-----------------+-------------+-------------+
|store_key|total_orders|total_quantity|totalgross_sales|total_discount|  total_sales|    average_sales|minimum_sales|maximum_sales|
+---------+------------+--------------+----------------+--------------+-------------+-----------------+-------------+-------------+
|     S004|       62319|        342712|      4903665600|  3.53716965E8|4.549948635E9|73010.61690656141|        640.0|     550000.0|
|     S001|       62865|        347260|      4961869800|   3.5782037E8| 4.60404943E9|73237.08629603118|        640.0|     550000.0|
|     S008|       62650|        344584|      4982859800|  3.59349485E8|4.623510315E9| 73799.0473264166|        640.0|     550000.0|
|     S002|       62534|        344707|      4997409500|  3.60713785E8|4.636695715E9|74146.79558320274|        640.0|     550000.0|
|     S005|       62468|        343455|      4929393600|   3.5669485E8| 4.57

In [ ]:
# 7. Create category-level summary.

# For every category calculate:

# total_quantity
# total_sales
# average_sales

category_level_summary_df = renamed_df.groupBy("category").agg(
    sum('quantity').alias("total_quantity"),
    sum('final_amount').alias("total_sales"),
    avg('final_amount').alias("average_sales")
)

category_level_summary_df.show()

+-----------+--------------+-------------+------------------+
|   category|total_quantity|  total_sales|     average_sales|
+-----------+--------------+-------------+------------------+
|Electronics|       1375312|2.96492138E10|118617.73192081807|
|  Furniture|        827651|6.520833225E9| 43401.33265666079|
|Accessories|        550146|   5.875369E8|5887.2022765759175|
+-----------+--------------+-------------+------------------+



In [ ]:
# 8. Create product-level summary.

# For every product calculate:
# total_quantity
# total_sales

# Sort products by highest sales.


product_level_summary_df = renamed_df.groupBy("product_name").agg(
    sum('quantity').alias("total_quantity"),
    sum('final_amount').alias("total_sales")
)


# withColumn()  --> if the column does not present in DF, it will create it.
      # otherwise the data will overwrite in existing column
# processed_sales_df = product_level_summary_df.withColumn("total_sales", format_number('total_sales', 2)).orderBy(desc("total_sales"))

processed_sales_df = product_level_summary_df.orderBy(desc("total_sales"))
processed_sales_df.show()

+------------+--------------+--------------+
|product_name|total_quantity|   total_sales|
+------------+--------------+--------------+
|      Laptop|        276617|1.411805175E10|
|Mobile Phone|        273946|   6.3521075E9|
|     Monitor|        273387|   4.5651987E9|
|     Printer|        276665|  3.84939825E9|
|Office Table|        275631|   3.0661866E9|
|Office Chair|        274072| 1.907108625E9|
|   Bookshelf|        277948|    1.547538E9|
|  Headphones|        274697|    7.644576E8|
|    Keyboard|        276175|    3.842613E8|
|       Mouse|        273971|    2.032756E8|
+------------+--------------+--------------+



In [ ]:
# 9. Write the processed transaction data.
# Output: processed_sales
# Format: CSV

processed_sales_df.write.mode('ignore').csv('processed_sales', header=True)

In [ ]:
csv_df= spark.read.csv('sales_data.csv', header= True)
csv_df.count()

csv_df.write.parquet('parquet_data')

470918

In [ ]:
input_df.write.parquet("parquet_sales_data")

In [ ]:
# create 3 new columns ==> year, month, date from order_date column

process_df = input_df.withColumn('year', year('order_date')) \
                      .withColumn('month', month('order_date')) \
                      .withColumn('day', day('order_date'))
process_df.printSchema()
process_df.select('order_date', 'year', 'month', 'day').show()

process_df.coalesce(1).write.mode('overwrite').partitionBy('year', 'month', 'day').parquet("multi_partition_sales_data")

process_df.coalesce(1).write.mode('overwrite').partitionBy('store_id','year', 'month', 'day').parquet("store_multi_partition_sales_data")


root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- sate: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)

+----------+----+-----+---+
|order_date|year|month|day|
+----------+----+-----+---+
|2026-05-31|2026|    5| 31|
|2026-03-07|2026|    3|  7|
|2026-03-29|2026|    3| 29|
|2026-02-24|2026|    2| 24|
|2026-06-20|2026|    6| 20|
|2026-04-09|2026|    4|  9|
|2026-01-07|2026|    1|  7|
|2026-01-27|2026|    1| 27

In [ ]:
# input_df.coalesce(1).write.bucketBy(5, 'product_id').saveAsTable("bucket_table")


spark.sql("select * from bucket_table limit 10").show()
spark.sql('create table sample(id int, name varchar(10))')

+----------+----------+-----------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+----------+------------+
|  order_id|order_date|customer_id|  customer_name|product_id|product_name|   category|store_id|    store_name|     city|       sate|quantity|unit_price|discount_pct|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+----------+------------+
|ORD0000003|2026-03-29|  CUST19584|  Daniel Martin|      P010|   Bookshelf|  Furniture|    S003|    City Store|  Chennai| Tamil Nadu|      10|      6000|           5|
|ORD0000007|2026-01-07|  CUST20498|      Ava White|      P004|     Monitor|Electronics|    S001| Central Store|Bengaluru|  Karnataka|       1|     18000|          20|
|ORD0000010|2026-01-22|  CUST14738|  Lucas Johnson|      P010|   Bookshelf|  Furniture|    S001| Central Store|Bengaluru|  Karnataka|       5|      6000|          20

DataFrame[]

In [ ]:
spark.sql("insert into sample values(1, 'charan')")
spark.sql('select * from sample').show()

spark.sql('drop table sample')

+---+------+
| id|  name|
+---+------+
|  1|charan|
|  1|charan|
+---+------+



DataFrame[]

In [ ]:
data = [('ram', 61), ('arun', 25), ('kiran', 16)]
columns =['name', 'age']

df = spark.createDataFrame(data, columns)
df.show()

# status --> age <18 minor, age b/w 18 and 60 adult, age >60 sr.citizen

df1= df.withColumn('status', expr(""" case when age < 18 then 'minor' when age between 18 and 60 then 'adult' else 'sr.citizen' end """))
df1.show()

+-----+---+
| name|age|
+-----+---+
|  ram| 61|
| arun| 25|
|kiran| 16|
+-----+---+

+-----+---+----------+
| name|age|    status|
+-----+---+----------+
|  ram| 61|sr.citizen|
| arun| 25|     adult|
|kiran| 16|     minor|
+-----+---+----------+



In [ ]:
# date functions


input_df = input_df.withColumn('year', year('order_date')) \
                      .withColumn('month', month('order_date')) \
                      .withColumn('day', day('order_date')) \
                      .withColumn('dayofmonth', dayofmonth('order_date')) \
                      .withColumn('day_of_week', dayofweek('order_date')) \
                      .withColumn('day_of_year', dayofyear('order_date')) \
                      .withColumn('year_month', date_format('order_date', 'yyyy-MM')) \
                      .withColumn('month_date', date_format('order_date', 'MM-dd')) \
                      .withColumn('date_month_year', date_format('order_date', 'dd-MM-yyyy')) \
                      .withColumn('date_month_year1', date_format('order_date', 'dd/MM/yyyy')) \
                      .withColumn('date_month_year2', date_format('order_date', 'dd.MM.yyyy')) \
                      .withColumn('year_month1', date_format('order_date', 'yyyy-MMM')) \
                      .withColumn('year_month2', date_format('order_date', 'yyyy-MMMM')) \
                      .withColumn('month_year', date_format('order_date', 'MMMM yyyy')) \



input_df.select('order_date', 'year', 'month', 'day', 'dayofmonth', 'day_of_week', 'day_of_year', 'year_month', 'month_date', 'date_month_year',
                'date_month_year1', 'date_month_year2', 'year_month1', 'year_month2', 'month_year').show()

# 1-> sunday 2 -> monday .... 7 -> saturday

+----------+----+-----+---+----------+-----------+-----------+----------+----------+---------------+----------------+----------------+-----------+-------------+-------------+
|order_date|year|month|day|dayofmonth|day_of_week|day_of_year|year_month|month_date|date_month_year|date_month_year1|date_month_year2|year_month1|  year_month2|   month_year|
+----------+----+-----+---+----------+-----------+-----------+----------+----------+---------------+----------------+----------------+-----------+-------------+-------------+
|2026-05-31|2026|    5| 31|        31|          1|        151|   2026-05|     05-31|     31-05-2026|      31/05/2026|      31.05.2026|   2026-May|     2026-May|     May 2026|
|2026-03-07|2026|    3|  7|         7|          7|         66|   2026-03|     03-07|     07-03-2026|      07/03/2026|      07.03.2026|   2026-Mar|   2026-March|   March 2026|
|2026-03-29|2026|    3| 29|        29|          1|         88|   2026-03|     03-29|     29-03-2026|      29/03/2026|      29

In [ ]:
input_df.withColumn('days_bw_order', datediff(current_date(), 'order_date'))  \
        .withColumn('months_bw_order', months_between(current_date(), 'order_date'))  \
        .withColumn('months_bw_order1', floor(months_between(current_date(), 'order_date')) )  \
        .withColumn('months_bw_order2', ceil(months_between(current_date(), 'order_date')) )  \
        .withColumn('months_bw_order_cast', months_between(current_date(), 'order_date').cast('int'))  \
        .withColumn('months_bw_order_round', round(months_between(current_date(), 'order_date'), 2))  \
        .select('order_date', 'days_bw_order', 'months_bw_order', 'months_bw_order1',  'months_bw_order2', 'months_bw_order_cast', 'months_bw_order_round').show()

+----------+-------------+---------------+----------------+----------------+--------------------+---------------------+
|order_date|days_bw_order|months_bw_order|months_bw_order1|months_bw_order2|months_bw_order_cast|months_bw_order_round|
+----------+-------------+---------------+----------------+----------------+--------------------+---------------------+
|2026-05-31|           96|     3.12903226|               3|               4|                   3|                 3.13|
|2026-03-07|          181|     5.90322581|               5|               6|                   5|                  5.9|
|2026-03-29|          159|     5.19354839|               5|               6|                   5|                 5.19|
|2026-02-24|          192|     6.35483871|               6|               7|                   6|                 6.35|
|2026-06-20|           76|     2.48387097|               2|               3|                   2|                 2.48|
|2026-04-09|          148|     4.8387096

In [ ]:
from datetime import date
df = spark.createDataFrame([(date.today(),)], ['joining_date'])

df.show()

df.withColumn('future_month', add_months('joining_date', 4)) \
  .withColumn('past_month', add_months('joining_date', -4)) \
  .withColumn('future_date', date_add('joining_date', 31)) \
  .withColumn('past_date', date_add('joining_date', -31)) \
  .withColumn('date_sub', date_sub('joining_date', 31)) \
  .withColumn('month_end', last_day('joining_date')) \
  .withColumn('month_start', trunc('joining_date', 'month') )\
  .withColumn('year_start', trunc('joining_date', 'year') )\
  .withColumn('next_monday', next_day('joining_date', 'Saturday') )\
  .withColumn('current_timestamp', current_timestamp() )\
  .withColumn('current_timestamp_india', from_utc_timestamp(current_timestamp(), 'Asia/Kolkata' )) \
  .withColumn('current_timestamp_india1', convert_timezone(None, lit('Asia/Kolkata'), current_timestamp()))\
  .show(10, False)

+------------+
|joining_date|
+------------+
|  2026-09-04|
+------------+

+------------+------------+----------+-----------+----------+----------+----------+-----------+----------+-----------+--------------------------+--------------------------+--------------------------+
|joining_date|future_month|past_month|future_date|past_date |date_sub  |month_end |month_start|year_start|next_monday|current_timestamp         |current_timestamp_india   |current_timestamp_india1  |
+------------+------------+----------+-----------+----------+----------+----------+-----------+----------+-----------+--------------------------+--------------------------+--------------------------+
|2026-09-04  |2027-01-04  |2026-05-04|2026-10-05 |2026-08-04|2026-08-04|2026-09-30|2026-09-01 |2026-01-01|2026-09-05 |2026-09-04 06:00:48.754471|2026-09-04 11:30:48.754471|2026-09-04 11:30:48.754471|
+------------+------------+----------+-----------+----------+----------+----------+-----------+----------+-----------+------

In [ ]:
# String functions

employee_data_diff_domains = [
    (201, "Liam Neeson", "liam.n@gmail.com", "Boston"),
    (202, "Olivia Wilde", "olivia.w@yahoo.com", "Miami"),
    (203, "Noah Centineo", "noah.c@outlook.com", "Seattle"),
    (204, "Emma Watson", "emma.w@protonmail.com", "San Francisco"),
    (205, "Oliver Queen", "oliver.q@icloud.com", "Star City"),
    (206, "Sophia Loren", "sophia.l@zoho.com", "Austin"),
    (207, "Elijah Wood", "elijah.w@aol.com", "Denver"),
    (208, "Amelia Earhart", "amelia.e@mail.com", "Kansas City"),
    (209, "Lucas Film", "lucas.f@yandex.com", "San Rafael"),
    (210, "Mia Khalifa", "mia.k@live.com", "Las Vegas")
]

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("email_id", StringType(), True),
    StructField("city", StringType(), True)
])

df = spark.createDataFrame(data=employee_data_diff_domains, schema=schema)
df.show(truncate=False)

+------+--------------+---------------------+-------------+
|emp_id|emp_name      |email_id             |city         |
+------+--------------+---------------------+-------------+
|201   |Liam Neeson   |liam.n@gmail.com     |Boston       |
|202   |Olivia Wilde  |olivia.w@yahoo.com   |Miami        |
|203   |Noah Centineo |noah.c@outlook.com   |Seattle      |
|204   |Emma Watson   |emma.w@protonmail.com|San Francisco|
|205   |Oliver Queen  |oliver.q@icloud.com  |Star City    |
|206   |Sophia Loren  |sophia.l@zoho.com    |Austin       |
|207   |Elijah Wood   |elijah.w@aol.com     |Denver       |
|208   |Amelia Earhart|amelia.e@mail.com    |Kansas City  |
|209   |Lucas Film    |lucas.f@yandex.com   |San Rafael   |
|210   |Mia Khalifa   |mia.k@live.com       |Las Vegas    |
+------+--------------+---------------------+-------------+



In [ ]:
df.withColumn('names', split('emp_name', ' ')) \
.withColumn('first_name', split('emp_name', ' ')[0]) \
.withColumn('last_names', split('emp_name', ' ')[1]) \
.withColumn('upper_name', upper('emp_name')) \
.withColumn('lower_name', lower('emp_name')) \
.withColumn('ltrim', ltrim('emp_name')) \
.withColumn('rtrim', rtrim('emp_name')) \
.withColumn('trim', trim('emp_name')) \
.withColumn('length', length('emp_name')) \
.show()

+------+--------------+--------------------+-------------+-----------------+----------+----------+--------------+--------------+--------------+--------------+--------------+------+
|emp_id|      emp_name|            email id|         city|            names|first_name|last_names|    upper_name|    lower_name|         ltrim|         rtrim|          trim|length|
+------+--------------+--------------------+-------------+-----------------+----------+----------+--------------+--------------+--------------+--------------+--------------+------+
|   201|   Liam Neeson|    liam.n@gmail.com|       Boston|   [Liam, Neeson]|      Liam|    Neeson|   LIAM NEESON|   liam neeson|   Liam Neeson|   Liam Neeson|   Liam Neeson|    11|
|   202|  Olivia Wilde|  olivia.w@yahoo.com|        Miami|  [Olivia, Wilde]|    Olivia|     Wilde|  OLIVIA WILDE|  olivia wilde|  Olivia Wilde|  Olivia Wilde|  Olivia Wilde|    12|
|   203| Noah Centineo|  noah.c@outlook.com|      Seattle| [Noah, Centineo]|      Noah|  Centin

In [ ]:
df.withColumn('user_name', substring_index('email_id', '@', 1)) \
  .withColumn('email_domain', substring_index('email_id', '@', -1)) \
  .withColumn('prefix', substring('email_id', 3,5)).show()

+------+--------------+--------------------+-------------+---------+--------------+------+
|emp_id|      emp_name|            email_id|         city|user_name|  email_domain|prefix|
+------+--------------+--------------------+-------------+---------+--------------+------+
|   201|   Liam Neeson|    liam.n@gmail.com|       Boston|   liam.n|     gmail.com| am.n@|
|   202|  Olivia Wilde|  olivia.w@yahoo.com|        Miami| olivia.w|     yahoo.com| ivia.|
|   203| Noah Centineo|  noah.c@outlook.com|      Seattle|   noah.c|   outlook.com| ah.c@|
|   204|   Emma Watson|emma.w@protonmail...|San Francisco|   emma.w|protonmail.com| ma.w@|
|   205|  Oliver Queen| oliver.q@icloud.com|    Star City| oliver.q|    icloud.com| iver.|
|   206|  Sophia Loren|   sophia.l@zoho.com|       Austin| sophia.l|      zoho.com| phia.|
|   207|   Elijah Wood|    elijah.w@aol.com|       Denver| elijah.w|       aol.com| ijah.|
|   208|Amelia Earhart|   amelia.e@mail.com|  Kansas City| amelia.e|      mail.com| elia.|

In [ ]:
# user_id ==> first 2 letters from first_name + first 2 letters from last_name + emp_id

df.withColumn('user_id', concat(substring(split('emp_name', ' ')[0], 1,2),  substring(split('emp_name', ' ')[1], 1,2) , 'emp_id')) \
  .withColumn('user_id_ws', concat_ws(' ',substring(split('emp_name', ' ')[0], 1,2),  substring(split('emp_name', ' ')[1], 1,2) , 'emp_id')) \
  .show()

# findout employees whose name started with 'L'
df.filter(col('emp_name').startswith('L')).show()

# findout employees whose name ends with 'N'
df.filter(col('emp_name').endswith('n')).show()

# findout employees who has gmail id's
df.filter(col('email_id').contains('gmail')).show()

# replace the city name "Star City" with 'Star'
df.withColumn('city', regexp_replace('city', 'Star City', 'Star')).show()



+------+--------------+--------------------+-------------+-------+----------+
|emp_id|      emp_name|            email_id|         city|user_id|user_id_ws|
+------+--------------+--------------------+-------------+-------+----------+
|   201|   Liam Neeson|    liam.n@gmail.com|       Boston|LiNe201| Li Ne 201|
|   202|  Olivia Wilde|  olivia.w@yahoo.com|        Miami|OlWi202| Ol Wi 202|
|   203| Noah Centineo|  noah.c@outlook.com|      Seattle|NoCe203| No Ce 203|
|   204|   Emma Watson|emma.w@protonmail...|San Francisco|EmWa204| Em Wa 204|
|   205|  Oliver Queen| oliver.q@icloud.com|    Star City|OlQu205| Ol Qu 205|
|   206|  Sophia Loren|   sophia.l@zoho.com|       Austin|SoLo206| So Lo 206|
|   207|   Elijah Wood|    elijah.w@aol.com|       Denver|ElWo207| El Wo 207|
|   208|Amelia Earhart|   amelia.e@mail.com|  Kansas City|AmEa208| Am Ea 208|
|   209|    Lucas Film|  lucas.f@yandex.com|   San Rafael|LuFi209| Lu Fi 209|
|   210|   Mia Khalifa|      mia.k@live.com|    Las Vegas|MiKh21

In [ ]:
employee_data_diff_domains = [
    (201, "Liam     Neeson", "liam.n@gmail.com@GMAIL.com", "Boston"),
    (202, "Olivia        Wilde", "olivia.w@yahoo.com", "Miami"),
    (203, "  Noah     Centineo", "noah.c@outlook.co", "Seattle"),
    (204, "Emma    Watson", "emma.w@protonmail.com", "San Francisco"),
    (205, "Oliver    Queen", "oliver.q@icloud.com", "Star City"),
    (206, "Sophia Loren", "sophia.l@zoho.com", "Austin"),
    (207, "Elijah   Wood", "elijah.w@aol.com", "Denver"),
    (208, "Amelia Earhart", "amelia.e@mail.com", "Kansas City"),
    (209, "Lucas    Film  ", "lucas.f@yandex.com", "San Rafael"),
    (210, "Mia Khalifa", "mia.k@live.com", "Las Vegas")
]

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("email_id", StringType(), True),
    StructField("city", StringType(), True)
])

df = spark.createDataFrame(data=employee_data_diff_domains, schema=schema)

df.withColumn('clean_name', regexp_replace(trim('emp_name'), '\\s+', ' ')).show()

+------+-------------------+--------------------+-------------+--------------+
|emp_id|           emp_name|            email_id|         city|    clean_name|
+------+-------------------+--------------------+-------------+--------------+
|   201|    Liam     Neeson|liam.n@gmail.com@...|       Boston|   Liam Neeson|
|   202|Olivia        Wilde|  olivia.w@yahoo.com|        Miami|  Olivia Wilde|
|   203|  Noah     Centineo|   noah.c@outlook.co|      Seattle| Noah Centineo|
|   204|     Emma    Watson|emma.w@protonmail...|San Francisco|   Emma Watson|
|   205|    Oliver    Queen| oliver.q@icloud.com|    Star City|  Oliver Queen|
|   206|       Sophia Loren|   sophia.l@zoho.com|       Austin|  Sophia Loren|
|   207|      Elijah   Wood|    elijah.w@aol.com|       Denver|   Elijah Wood|
|   208|     Amelia Earhart|   amelia.e@mail.com|  Kansas City|Amelia Earhart|
|   209|    Lucas    Film  |  lucas.f@yandex.com|   San Rafael|    Lucas Film|
|   210|        Mia Khalifa|      mia.k@live.com|   

In [ ]:
# regexp_extract()  --> (column, pattern, group_number)

df.withColumn('domain', regexp_extract('email_id', r'@([A-Za-z0-9-]+\.com)$', 1)).show()

+------+-------------------+--------------------+-------------+--------------+
|emp_id|           emp_name|            email_id|         city|        domain|
+------+-------------------+--------------------+-------------+--------------+
|   201|    Liam     Neeson|liam.n@gmail.com@...|       Boston|     GMAIL.com|
|   202|Olivia        Wilde|  olivia.w@yahoo.com|        Miami|     yahoo.com|
|   203|  Noah     Centineo|   noah.c@outlook.co|      Seattle|              |
|   204|     Emma    Watson|emma.w@protonmail...|San Francisco|protonmail.com|
|   205|    Oliver    Queen| oliver.q@icloud.com|    Star City|    icloud.com|
|   206|       Sophia Loren|   sophia.l@zoho.com|       Austin|      zoho.com|
|   207|      Elijah   Wood|    elijah.w@aol.com|       Denver|       aol.com|
|   208|     Amelia Earhart|   amelia.e@mail.com|  Kansas City|      mail.com|
|   209|    Lucas    Film  |  lucas.f@yandex.com|   San Rafael|    yandex.com|
|   210|        Mia Khalifa|      mia.k@live.com|   

In [ ]:
data =[(1, '+91-1234567890'), (2, '+91-9876543211'), (3, '9876543210'), (4, '+1-324567890'), (5, '+1-323424567890')]

df = spark.createDataFrame(data, ['id', 'number'])


# get actual 10 digit mobile number(without country code)

df.withColumn('phone_number', regexp_extract('number', r'(\d{10})$', 1) ).show()

+---+---------------+------------+
| id|         number|phone_number|
+---+---------------+------------+
|  1| +91-1234567890|  1234567890|
|  2| +91-9876543211|  9876543211|
|  3|     9876543210|  9876543210|
|  4|   +1-324567890|            |
|  5|+1-323424567890|  3424567890|
+---+---------------+------------+



In [ ]:
data =[(1, 'order id is: ORD-001ch23'), (2, 'order id is: ORD-002'), (3, 'order id is: ORD-003')]
df = spark.createDataFrame(data, ['id', 'description'])

df.show()

# extract  ORD-001, ORD-002
df.withColumn('order_id', regexp_extract('description', r'(ORD-\d+)',1)).show()

+---+--------------------+
| id|         description|
+---+--------------------+
|  1|order id is: ORD-...|
|  2|order id is: ORD-002|
|  3|order id is: ORD-003|
+---+--------------------+

+---+--------------------+--------+
| id|         description|order_id|
+---+--------------------+--------+
|  1|order id is: ORD-...| ORD-001|
|  2|order id is: ORD-002| ORD-002|
|  3|order id is: ORD-003| ORD-003|
+---+--------------------+--------+

